# PhiSat-2 Search and Download

This notebook shows a minimal PhiSat-2 workflow with the new `phidown` provider support:

1. resolve the local repository and credential file
2. search INSULA platform files with `PhiSat2Searcher.query()`
3. inspect the matching results
4. optionally download one selected product

The notebook is safe to execute without credentials. Live search and download cells are skipped until you provide a valid `.s5cfg` file with a `[phisat2]` section and a real search token.

In [1]:
from __future__ import annotations

import configparser
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in [cwd, cwd.parent] if (candidate / "phidown").exists()),
    cwd,
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import phidown
from phidown import PhiSat2Searcher

print(f"Repo root: {repo_root}")
print(f"phidown version: {phidown.__version__}")

Repo root: /Users/roberto.delprete/Downloads/phidown
phidown version: 0.1.27


## Credentials

PhiSat-2 uses the shared `.s5cfg` file. Insert the `[phisat2]` block directly below the existing `[default]` block:

```ini
[phisat2]
username = your_email@example.com
password = your_password
base_url = https://phisat2.insula.earth
api_base = https://phisat2.insula.earth/secure/api/v2.0
authorization_endpoint = https://identity.insula.earth/realms/phisat2/protocol/openid-connect/auth
token_endpoint = https://identity.insula.earth/realms/phisat2/protocol/openid-connect/token
redirect_uri = http://localhost:9207/auth
client_id = api-client
```

You can keep the file at the repository root as `./.s5cfg`, or pass another path through the parameter cell below.

In [2]:
# Notebook parameters
config_candidates = [repo_root / ".s5cfg", cwd / ".s5cfg"]
CONFIG_FILE = next((path for path in config_candidates if path.exists()), repo_root / ".s5cfg")

# Replace SET_ME or export PHISAT2_NOTEBOOK_FILTER before execution.
SEARCH_FILTER = os.getenv("PHISAT2_NOTEBOOK_FILTER", "SET_ME")
RESULTS_PER_PAGE = 10

# Set PHISAT2_NOTEBOOK_DOWNLOAD=1 to enable the live download cell.
DOWNLOAD_ENABLED = os.getenv("PHISAT2_NOTEBOOK_DOWNLOAD", "0") == "1"
DOWNLOAD_INDEX = 0
DOWNLOAD_DIR = repo_root / "notebooks" / "data" / "phisat2"

print(f"CONFIG_FILE={CONFIG_FILE}")
print(f"SEARCH_FILTER={SEARCH_FILTER!r}")
print(f"RESULTS_PER_PAGE={RESULTS_PER_PAGE}")
print(f"DOWNLOAD_ENABLED={DOWNLOAD_ENABLED}")
print(f"DOWNLOAD_DIR={DOWNLOAD_DIR}")

CONFIG_FILE=/Users/roberto.delprete/Downloads/phidown/.s5cfg
SEARCH_FILTER='SET_ME'
RESULTS_PER_PAGE=10
DOWNLOAD_ENABLED=False
DOWNLOAD_DIR=/Users/roberto.delprete/Downloads/phidown/notebooks/data/phisat2


In [3]:
def has_phisat2_section(config_path: Path) -> bool:
    if not config_path.exists():
        return False
    parser = configparser.ConfigParser()
    parser.read(config_path)
    return parser.has_section("phisat2")


phisat2_ready = has_phisat2_section(CONFIG_FILE)
search_ready = phisat2_ready and SEARCH_FILTER != "SET_ME"

if not CONFIG_FILE.exists():
    print(f"Config file not found: {CONFIG_FILE}")
elif not phisat2_ready:
    print(f"Config file exists but is missing a [phisat2] section: {CONFIG_FILE}")
else:
    print(f"Found PhiSat-2 credentials in: {CONFIG_FILE}")

if SEARCH_FILTER == "SET_ME":
    print("SEARCH_FILTER is still SET_ME. Live search will be skipped.")
else:
    print(f"Live search is armed with filter: {SEARCH_FILTER}")

Config file not found: /Users/roberto.delprete/Downloads/phidown/.s5cfg
SEARCH_FILTER is still SET_ME. Live search will be skipped.


In [4]:
searcher = None
results = pd.DataFrame()

if search_ready:
    searcher = PhiSat2Searcher(config_file=str(CONFIG_FILE))
    results = searcher.query(SEARCH_FILTER, results_per_page=RESULTS_PER_PAGE)
    print(f"Found {len(results)} matching PhiSat-2 products")
    if results.empty:
        print("No products matched the provided filter.")
    else:
        display_columns = [
            column
            for column in ["Id", "Name", "ContentLength", "CreatedAt", "DownloadUrl"]
            if column in results.columns
        ]
        display(results[display_columns].head(min(len(results), RESULTS_PER_PAGE)))
else:
    print(
        "Skipping live PhiSat-2 search. Provide a valid .s5cfg [phisat2] section and "
        "set SEARCH_FILTER (or PHISAT2_NOTEBOOK_FILTER) to run this cell against INSULA."
    )

Skipping live PhiSat-2 search. Provide a valid .s5cfg [phisat2] section and set SEARCH_FILTER (or PHISAT2_NOTEBOOK_FILTER) to run this cell against INSULA.


In [5]:
selected_product = None

if not results.empty:
    if DOWNLOAD_INDEX >= len(results):
        raise IndexError(f"DOWNLOAD_INDEX={DOWNLOAD_INDEX} is out of range for {len(results)} results")
    selected_product = results.iloc[DOWNLOAD_INDEX]
    print(f"Selected row {DOWNLOAD_INDEX}")
    display(selected_product[[column for column in ["Id", "Name", "DownloadUrl"] if column in results.columns]])
else:
    print("No selected product yet because the search step did not produce any rows.")

No selected product yet because the search step did not produce any rows.


In [6]:
download_path = None

if DOWNLOAD_ENABLED and selected_product is not None and searcher is not None:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    download_path = searcher.download_product(
        product_id=selected_product["Id"],
        output_dir=str(DOWNLOAD_DIR),
        file_name=selected_product.get("Name"),
        show_progress=True,
    )
    print(f"Downloaded to: {download_path}")
else:
    print(
        "Download is disabled by default. Set PHISAT2_NOTEBOOK_DOWNLOAD=1 after verifying the "
        "search results if you want this notebook to fetch the selected product."
    )

Download is disabled by default. Set PHISAT2_NOTEBOOK_DOWNLOAD=1 after verifying the search results if you want this notebook to fetch the selected product.


## Next steps

- Replace `SEARCH_FILTER = "SET_ME"` with a real session ID or filename fragment.
- Confirm the top rows look correct.
- Enable the download cell with `PHISAT2_NOTEBOOK_DOWNLOAD=1`.
- Re-run the notebook once credentials and the search token are in place.